<a href="https://colab.research.google.com/github/AshishPal20/Machine-Learning-/blob/main/Real_or_FakeNews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing the depedency



In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

Printing the stopwords(stopwords are the words that does not add that much value to our text)

In [ ]:
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# DATA PREPROCESSING

In [ ]:
news_dataset = pd.read_csv(
    '/content/FakeNewsNet.csv',
    engine='python',
    on_bad_lines='skip',
    encoding='latin1'
)

In [ ]:
news_dataset.shape

(23196, 5)

In [ ]:
#print first 5 rows of dataset
x=news_dataset.head()
print(x)

                                               title  \
0  Kandi Burruss Explodes Over Rape Accusation on...   
1  People's Choice Awards 2018: The best red carp...   
2  Sophia Bush Sends Sweet Birthday Message to 'O...   
3  Colombian singer Maluma sparks rumours of inap...   
4  Gossip Girl 10 Years Later: How Upper East Sid...   

                                            news_url        source_domain  \
0  http://toofab.com/2017/05/08/real-housewives-a...           toofab.com   
1  https://www.today.com/style/see-people-s-choic...        www.today.com   
2  https://www.etonline.com/news/220806_sophia_bu...     www.etonline.com   
3  https://www.dailymail.co.uk/news/article-33655...  www.dailymail.co.uk   
4  https://www.zerchoo.com/entertainment/gossip-g...      www.zerchoo.com   

   tweet_num  real  
0         42     1  
1          0     1  
2         63     1  
3         20     1  
4         38     1  


In [ ]:
# count the number of missing values in dataset
news_dataset.isnull().sum()

,0
title,0
news_url,330
source_domain,330
tweet_num,0
real,0


In [ ]:
# replacing the null values with empty string
news_dataset = news_dataset.fillna('')

In [ ]:
# merging the title and text of the news dataset
news_dataset['content'] = news_dataset['title']+' '+news_dataset['news_url']
print(news_dataset['content'])

0        Kandi Burruss Explodes Over Rape Accusation on...
1        People's Choice Awards 2018: The best red carp...
2        Sophia Bush Sends Sweet Birthday Message to 'O...
3        Colombian singer Maluma sparks rumours of inap...
4        Gossip Girl 10 Years Later: How Upper East Sid...
                               ...                        
23191    Pippa Middleton wedding: In case you missed it...
23192    Zayn Malik & Gigi Hadidâs Shocking Split: Wh...
23193    Jessica Chastain Recalls the Moment Her Mother...
23194    Tristan Thompson Feels "Dumped" After KhloÃ© K...
23195    Kelly Clarkson Performs a Medley of Kendrick L...
Name: content, Length: 23196, dtype: object


In [ ]:
# separate the data and label
X = news_dataset.drop(columns='real', axis=1)
Y = news_dataset['real']

In [ ]:
print(X)
print(Y)

                                                   title  \
0      Kandi Burruss Explodes Over Rape Accusation on...   
1      People's Choice Awards 2018: The best red carp...   
2      Sophia Bush Sends Sweet Birthday Message to 'O...   
3      Colombian singer Maluma sparks rumours of inap...   
4      Gossip Girl 10 Years Later: How Upper East Sid...   
...                                                  ...   
23191  Pippa Middleton wedding: In case you missed it...   
23192  Zayn Malik & Gigi Hadidâs Shocking Split: Wh...   
23193  Jessica Chastain Recalls the Moment Her Mother...   
23194  Tristan Thompson Feels "Dumped" After KhloÃ© K...   
23195  Kelly Clarkson Performs a Medley of Kendrick L...   

                                                news_url  \
0      http://toofab.com/2017/05/08/real-housewives-a...   
1      https://www.today.com/style/see-people-s-choic...   
2      https://www.etonline.com/news/220806_sophia_bu...   
3      https://www.dailymail.co.uk/news

# STEMMING PROCEDURE
Stemming is the process of reducing a word to its Root word

example: actor, actress, acting --> act

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stem(content):
  stemmed_content = re.sub('[^a-zA-Z]',' ',content)
  stemmed_content = stemmed_content.lower()
  stemmed_content = stemmed_content.split()
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
  stemmed_content = ' '.join(stemmed_content)
  return stemmed_content

In [ ]:
news_dataset['content'] = news_dataset['content'].apply(stem)


In [ ]:
print(news_dataset['content'])

0        kandi burruss explod rape accus real housew at...
1        peopl choic award best red carpet look http ww...
2        sophia bush send sweet birthday messag one tre...
3        colombian singer maluma spark rumour inappropr...
4        gossip girl year later upper east sider shock ...
                               ...                        
23191    pippa middleton wed case miss pippa marri lace...
23192    zayn malik gigi hadid shock split chanc reunit...
23193    jessica chastain recal moment mother boyfriend...
23194    tristan thompson feel dump khlo kardashian ref...
23195    kelli clarkson perform medley kendrick lamar h...
Name: content, Length: 23196, dtype: object


In [ ]:
# Separate data and label
X = news_dataset['content'].values
Y = news_dataset['real'].values


In [ ]:
print(X)
print(Y)

['kandi burruss explod rape accus real housew atlanta reunion video http toofab com real housew atlanta kandi burruss rape phaedra park porsha william'
 'peopl choic award best red carpet look http www today com style see peopl choic award red carpet look'
 'sophia bush send sweet birthday messag one tree hill co star hilari burton breyton eva http www etonlin com news sophia bush send sweet birthday messag one tree hill co star hilari burton breyton eva'
 ...
 'jessica chastain recal moment mother boyfriend slap kick genit http www justjar com jessica chastain recal moment mother boyfriend slap kick genit'
 'tristan thompson feel dump khlo kardashian refus let move la home exclus www intouchweekli com post tristan thompson feel dump khloe kardashian'
 'kelli clarkson perform medley kendrick lamar humbl hit billboard music award http www billboard com articl news bbma kelli clarkson medley bbma']
[1 1 1 ... 1 0 1]


In [ ]:
# Converting textual data to numerical value
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [ ]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 316924 stored elements and shape (23196, 17314)>
  Coords	Values
  (0, 175)	0.14434726829473674
  (0, 896)	0.3389455172701989
  (0, 2218)	0.4346705393896895
  (0, 3144)	0.028363875555072016
  (0, 5230)	0.19914481139678233
  (0, 7204)	0.27967207824608226
  (0, 7240)	0.03029315079920102
  (0, 8136)	0.4346705393896895
  (0, 11301)	0.16947275863509945
  (0, 11540)	0.2113369959982094
  (0, 11846)	0.21414819505876584
  (0, 12356)	0.33456306448701023
  (0, 12418)	0.2501629274621897
  (0, 12736)	0.1394415242605404
  (0, 15510)	0.1652746695506423
  (0, 16345)	0.1037611672890413
  (0, 16828)	0.12458641416073078
  (1, 985)	0.33310329340670436
  (1, 1532)	0.18067383197494744
  (1, 2474)	0.4135533362491038
  (1, 2851)	0.4642049540481571
  (1, 3144)	0.04487540286284784
  (1, 7240)	0.04792777148735079
  (1, 9039)	0.36163482558348325
  (1, 11465)	0.25473967851260193
  :	:
  (23194, 8547)	0.17169083170481492
  (23194, 8810)	0.211463587763149

Splitting the data to training and test data

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify=Y, random_state=2)

Training the model: Logistic regression model

In [ ]:
model = LogisticRegression()

In [ ]:
model.fit(X_train, Y_train)

LogisticRegression()

# Evaluation

In [ ]:
# Accuracy score on training data
X_train_prediction = model.predict(X_train)
training_data = accuracy_score(X_train_prediction, Y_train)

In [ ]:
print('Accuracy score of training data', training_data)

Accuracy score of training data 0.9483724940719983


In [ ]:
# Accuracy score on test data
X_test_prediction = model.predict(X_test)
test_data = accuracy_score(X_test_prediction, Y_test)

In [ ]:
print('Accuracy on test data', test_data)

Accuracy on test data 0.9269396551724138


Making a predictive system

In [ ]:
X_new = X_test[0]

prediction = model.predict(X_new)
print(prediction)

if(prediction[0]==1):
  print('The news is real')
else:
  print('The news is fake')

[1]
The news is real


In [ ]:
print(Y_test[10])

0
